In [1]:
import requests
import json
import os
from datetime import datetime
import time

In [8]:
#Storing the first URL in a id variable and using the placeholder {} fetching each story details.
id = "https://hacker-news.firebaseio.com/v0/topstories.json"
story_item = "https://hacker-news.firebaseio.com/v0/item/{}.json"

headers = {"User-Agent": "TrendPulse/1.0"}

#Created a dictionary to store the corresponding keywords in its respective category.
categories = {
    "technology":["AI", "software", "tech", "code", "computer", "data", "cloud", "API", "GPU", "LLM"],
    "worldnews" :["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports" : ["NFL", "NBA", "FIFA", "sport", "game", "team", "player", "league", "championship"],
    "science" : ["research", "study", "space", "physics", "biology", "discovery", "NASA", "genome"],
    "entertainment" : ["movie", "film", "music", "Netflix", "game", "book", "show", "award", "streaming"]
}

# Storing collected stories is a new list
collected_stories = []

# To track how many stories are fetched from each category and initially storing 0 stories fetched
category_count = {}
for category in categories:
    category_count[category] = 0

print(category_count)


#created a function to fetch all the top 500 story id's by using the .get method.  
def fetch_top_story_ids():
    try:
        response = requests.get(id, headers=headers) #Here we are
        response.raise_for_status()#used the .raise_for_status to check whether we have any error while fetching the data.
        data  = response.json()#converting the response into a json format
        return data[0:500] #fetching the first 500 id's
    except Exception as e:
        print(f"Error fetching top stories: {e}")
        return []


#created a function to fetch all the individual stories for each story_id
def fetch_story(story_id):
    try:
        response = requests.get(story_item.format(story_id), headers=headers)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching story {story_id}: {e}")
        return None
    


# #This function Looks at ONE story and checks Which category does this story belong to?”
def categorize_story(title):
    title_lower = title.lower() #to check the case-insensitive lowering the title using the .lower() function

    #looping through the categories dictionary by separating and storing in category and keywords.
    for category, keywords in categories.items():
        for keyword in keywords: #looping over the keywords list and iterating over each keyword in the list.
            if keyword in title_lower: ##now just checking whether the keyword in the categoriers dictionary is present in title_lower if yes then just return the categrory.
                return category
    return None


#Calling the fetch_top_story_ids() function. and storing the dictionary in story_ids variable

story_ids = fetch_top_story_ids()

#What this loop does controls collection per category It ensures: 25 tech stories, 25 sports stories, etc...
for category in categories:
    print(f"\nProcessing category: {category}")

    for story_id in story_ids: #Looping though the story_ids dictionary

        # Stop if category limit reached
        if category_count[category] >= 25:
            break

        story = fetch_story(story_id) #calling the fetch_story() function and giving the current story_id as the argument.

        if not story: #If no story is found then continue to the next story_id
            continue

        title = story.get("title", "")#using the .get() method we are catching the "title" key from the story json.

        # Skip if no title
        if not title:
            continue

        detected_category = categorize_story(title)#Calling the categorize_story() function and giving the title as the argument which gives us 

        # Only collect if it matches current category
        if detected_category == category:

            story_data = {
                    "post_id": story.get("id"),
                    "title": title,
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),
                    "collected_at": datetime.now().isoformat()
                }

            collected_stories.append(story_data)
            category_count[category] += 1

        # Sleep AFTER finishing one category (IMPORTANT RULE)
        time.sleep(2)


# Create data folder if not exists
if not os.path.exists("data"):
    os.makedirs("data")

# File name with date
filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

# Save JSON file
with open(filename, "w", encoding="utf-8") as f:
    json.dump(collected_stories, f, indent=4)

print(f"\nCollected {len(collected_stories)} stories. Saved to {filename}")

    



















{'technology': 0, 'worldnews': 0, 'sports': 0, 'science': 0, 'entertainment': 0}

Processing category: technology

Processing category: worldnews

Processing category: sports

Processing category: science

Processing category: entertainment

Collected 92 stories. Saved to data/trends_20260414.json
